# 作业3 · 方向B：用微导纳米的财报训练一个「董秘」

**思路**：把「投资者提问 → 董秘作答」的问答对喂给 Qwen2.5-1.5B 做 LoRA 微调（SFT），
让它学会董秘的**口径与格式**（先结论、再引原文、最后标出处），
再用四格对照量出它到底学会了什么、没学会什么。

**跑之前**：菜单 `代码执行程序/Runtime → 更改运行时类型/Change runtime type → 硬件加速器选 T4 GPU`。

**需要上传三个文件**（都在 `colab_upload/` 里）：
- `train.jsonl`：104 条训练问答（chat 格式）
- `eval.jsonl`：四格评测数据（`split`=train/unseen，`q` 原问法，`q_alt` 换问法，`ref_a` 参考答案，`ref_nums` 关键数字，`page`）
- `kb.jsonl`：2219 块知识库（给 A 方案做 BM25 检索）


In [ ]:
# 0. 确认 GPU
!nvidia-smi
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "没有 GPU，请先改运行时类型")


In [ ]:
# 1. 装 unsloth（2~4 分钟，转圈正常）
%%capture
!pip install unsloth
print("安装完成")


In [ ]:
# 2. 上传三个文件：train.jsonl / eval.jsonl / kb.jsonl（一起选）
from google.colab import files
uploaded = files.upload()
print("已上传:", list(uploaded.keys()))


In [ ]:
# 3. 加载小模型（纯文本 Qwen2.5-1.5B-Instruct，4bit 量化）
from unsloth import FastLanguageModel
import torch, gc

try:
    del model
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()

max_seq_length = 2048
CANDIDATES = ["unsloth/Qwen2.5-1.5B-Instruct", "unsloth/Qwen2.5-3B-Instruct"]
model = tokenizer = MODEL = None
for name in CANDIDATES:
    try:
        print("尝试加载", name)
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=name, max_seq_length=max_seq_length, load_in_4bit=True, dtype=None)
        MODEL = name
        break
    except Exception as e:
        print("  失败:", type(e).__name__, str(e)[:200])
if model is None:
    raise SystemExit("模型没加载成功，请把报错发给助教")
print("已加载:", MODEL)


In [ ]:
# 4. 装 LoRA
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0.0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
    max_seq_length=max_seq_length)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"可训练参数 {trainable:,} / {total:,} = {trainable/total*100:.2f}%")


In [ ]:
# 5. 准备训练数据
import json
from datasets import Dataset

rows = [json.loads(l) for l in open("train.jsonl", encoding="utf-8")]
def to_text(ex):
    return {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)}
ds = Dataset.from_list(rows).map(to_text, remove_columns=["messages"])
print(f"训练样本 {len(ds)} 条")
print(ds[0]["text"][:500])


In [ ]:
# 6. 训练（3 轮，3~6 分钟）
from trl import SFTTrainer
try:
    from trl import SFTConfig
    args = SFTConfig(
        dataset_text_field="text", max_seq_length=max_seq_length,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=3, learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407, output_dir="outputs", report_to="none")
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=args)
except Exception as e:
    print("退回 TrainingArguments：", type(e).__name__)
    from transformers import TrainingArguments
    args = TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=3, learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407, output_dir="outputs", report_to="none")
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds,
                         args=args, dataset_text_field="text", max_seq_length=max_seq_length)
print(trainer.train())


In [ ]:
# 7. 生成 + 判分工具
# 判分口径：
#   数字命中率 = 参考答案里的关键数字，预测里出现了多少
#   答对       = 数字命中率 >= 0.6
#   引用正确   = 预测里写对了"第 X 页"（格式学没学会）
FastLanguageModel.for_inference(model)

SYSTEM = ("你是江苏微导纳米科技股份有限公司（科创板 688147）的董事会秘书办公室助理。"
          "只依据财报回答投资者提问：先给结论，再用「」原样引用原文，最后一行写 依据：[报告 第X页]；"
          "材料里没有的一律写“报告未披露”，绝不编造数字。")

def gen(question, context=None, max_new_tokens=360):
    user = question if not context else f"【财报原文片段】\n{context}\n\n投资者提问：{question}"
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def norm(s):
    return s.replace(" ", "").replace("\u3000", "")

def score(pred, item):
    # 修正后的判分口径（回应批改意见）：
    #   有数字  → 关键数字命中 >= 60% 才算对
    #   无数字  → 不再用 8 字片段重合兜底（会把"说得像"算成答对），只认「引用页码正确」
    np_ = norm(pred).replace(",", "")
    nums = item.get("ref_nums") or []
    page = str(item.get("page", ""))
    cite = 1 if f"第{page}页" in np_ else 0
    quote = 1 if ("「" in pred) or ("“" in pred) else 0
    if nums:
        hit = sum(1 for n in nums if n in np_) / len(nums)
        ok = 1 if hit >= 0.6 else 0
    else:
        hit = float("nan")
        ok = cite          # 没有可比数字时，只有引对了页码才算对
    return {"数字命中率": hit, "答对": ok, "引用正确": cite, "带引文": quote}

print(gen("公司报告期内研发人员数量占员工总数的比例是多少？"))


In [ ]:
# 8. 四格评测：训练事实 / 未见事实 × 原问法 / 换问法
import pandas as pd

items = [json.loads(l) for l in open("eval.jsonl", encoding="utf-8")]
print("评测集：", len(items), "条（train", sum(1 for x in items if x["split"]=="train"),
      "/ unseen", sum(1 for x in items if x["split"]=="unseen"), "）")

results = []
for it in items:
    for mode, q in (("原问法", it["q"]), ("换问法", it["q_alt"])):
        pred = gen(q)
        results.append({"split": it["split"], "mode": mode, "chunk_id": it["chunk_id"],
                        "page": it["page"], "q": q, "pred": pred, "ref": it["ref_a"],
                        **score(pred, it)})
        print(".", end="", flush=True)
df = pd.DataFrame(results)
df.to_csv("sft_results.csv", index=False, encoding="utf-8-sig")
print("\n已保存 sft_results.csv")

grid = (df.groupby(["split", "mode"])
          .agg(样本数=("答对", "size"), 答对率=("答对", "mean"),
               引用正确率=("引用正确", "mean"), 数字命中率=("数字命中率", "mean"))
          .round(3))
print("\n===== LoRA 微调后 · 四格对照 =====")
print(grid)
df.to_pickle("sft_df.pkl")


In [ ]:
# 9. A 方案对照：BM25 检索 + 同一个基座模型（不微调）
!pip install -q jieba rank_bm25
import json, jieba
from rank_bm25 import BM25Okapi
from unsloth import FastLanguageModel as FLM

kb = [json.loads(l) for l in open("kb.jsonl", encoding="utf-8")]
corpus = [list(jieba.cut(c["text"])) for c in kb]
bm25 = BM25Okapi(corpus)
print("知识库块数:", len(kb))

base, base_tok = FLM.from_pretrained(model_name=MODEL, max_seq_length=max_seq_length,
                                     load_in_4bit=True, dtype=None)
FLM.for_inference(base)

def gen_base(question, context, max_new_tokens=400):
    user = (f"【财报原文片段】\n{context}\n\n"
            f"请只依据上面的原文回答投资者提问：{question}\n"
            "要求：第一句给结论（带关键数字），第二句用「」引用原文，最后一行写 依据：[报告 第X页]；"
            "原文里没有答案就写：报告未披露。")
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    prompt = base_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = base_tok(prompt, return_tensors="pt").to("cuda")
    out = base.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                        pad_token_id=base_tok.eos_token_id)
    text = base_tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if len(text) < 30:      # 只吐了个"依据"行 → 用更直白的问法重试一次
        msgs2 = [{"role": "user", "content":
                  f"根据下面的财报原文回答问题：{question}\n\n原文：\n{context}\n\n请给出完整回答。"}]
        p2 = base_tok.apply_chat_template(msgs2, tokenize=False, add_generation_prompt=True)
        i2 = base_tok(p2, return_tensors="pt").to("cuda")
        o2 = base.generate(**i2, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=base_tok.eos_token_id)
        text = base_tok.decode(o2[0][i2["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return text

rows_rag = []
unseen = [x for x in items if x["split"] == "unseen"]
for it in unseen:
    idx = bm25.get_top_n(list(jieba.cut(it["q"])), range(len(kb)), n=3)
    ctx = "\n\n".join(kb[i]["text"] for i in idx)
    pred = gen_base(it["q"], ctx)
    rows_rag.append({"chunk_id": it["chunk_id"], "page": it["page"],
                     "recalled": [kb[i]["id"] for i in idx], "pred": pred,
                     **score(pred, it)})
    print(".", end="", flush=True)
rag = pd.DataFrame(rows_rag)
rag.to_csv("rag_results.csv", index=False, encoding="utf-8-sig")
print("\n\n===== A 方案（BM25 + 基座，未微调）· 未见事实 =====")
print("答对率 %.1f%%　引用正确率 %.1f%%　数字命中率 %.3f" % (
    rag["答对"].mean()*100, rag["引用正确"].mean()*100, rag["数字命中率"].mean()))


In [ ]:
# 10. 总表 + 下载
import pandas as pd
rows = []
for split, label in (("train", "训练事实"), ("unseen", "未见事实")):
    for mode in ("原问法", "换问法"):
        sub = df[(df["split"] == split) & (df["mode"] == mode)]
        if len(sub):
            rows.append({"方案": f"微调后 · {label} · {mode}", "样本数": len(sub),
                         "答对率": sub["答对"].mean(),
                         "引用正确率": sub["引用正确"].mean(),
                         "数字命中率": sub["数字命中率"].mean()})
rows.append({"方案": "A 基线：BM25 + 基座（未微调）· 未见事实", "样本数": len(rag),
             "答对率": rag["答对"].mean(), "引用正确率": rag["引用正确"].mean(),
             "数字命中率": rag["数字命中率"].mean()})

summary = pd.DataFrame(rows).round(3)
summary["答对率"] = (summary["答对率"] * 100).round(1).astype(str) + "%"
summary["引用正确率"] = (summary["引用正确率"] * 100).round(1).astype(str) + "%"
print(summary.to_string(index=False))
summary.to_csv("对照表.csv", index=False, encoding="utf-8-sig")

from google.colab import files as colab_files
for f in ["对照表.csv", "sft_results.csv", "rag_results.csv"]:
    try:
        colab_files.download(f)
    except Exception as e:
        print("下载", f, "失败:", e)


In [ ]:
# 补 1. 保存训练日志（老师要「可核对的训练日志」）—— 等训练跑完再执行这一格
import pandas as pd
hist = trainer.state.log_history
log_df = pd.DataFrame(hist)
log_df.to_csv("train_log.csv", index=False, encoding="utf-8-sig")
print(f"训练日志 {len(log_df)} 行 -> train_log.csv")
loss_rows = [h for h in hist if "loss" in h]
print("=== loss 前 5 条 ===")
for h in loss_rows[:5]:
    print(h)
print("=== loss 后 5 条 ===")
for h in loss_rows[-5:]:
    print(h)
print("\n可训练参数:", f"{trainable:,}", "/", f"{total:,}", f"= {trainable/total*100:.2f}%")
from google.colab import files as cf
cf.download("train_log.csv")


In [ ]:
# 补 2. 四格样例：每格 2 条，给出「问题 / SFT 回答 / 同一基座 RAG 回答 / 判定」
#     等第 9 格（A 基线）跑完再执行；这一格是老师明确要求的核对材料
import pandas as pd
import jieba

def rag_answer(q, topk=3):
    idx = bm25.get_top_n(list(jieba.cut(q)), range(len(kb)), n=topk)
    ctx = "\n\n".join(kb[i]["text"] for i in idx)
    return ctx, [kb[i]["id"] for i in idx]

item_by_chunk = {}
for x in items:
    item_by_chunk.setdefault(x["chunk_id"], x)

rows = []
for split in ("train", "unseen"):
    for mode in ("原问法", "换问法"):
        sub = df[(df["split"] == split) & (df["mode"] == mode)].head(2)
        for _, row in sub.iterrows():
            it = item_by_chunk.get(row["chunk_id"], {})
            ctx, recalled = rag_answer(row["q"])
            rag_pred = gen_base(row["q"], ctx)
            rs = score(rag_pred, it)
            rows.append({
                "格子": ("训练事实" if split == "train" else "未见事实") + "·" + mode,
                "chunk_id": row["chunk_id"], "出处页码": row["page"],
                "问题": row["q"],
                "参考答案": row["ref"][:200],
                "SFT回答": row["pred"][:300],
                "SFT判定": f"数字命中{row['数字命中率']} 答对{row['答对']} 引用{row['引用正确']}",
                "RAG回答": rag_pred[:300],
                "RAG判定": f"数字命中{rs['数字命中率']} 答对{rs['答对']} 引用{rs['引用正确']}",
                "RAG召回": ",".join(recalled),
            })
samp = pd.DataFrame(rows)
samp.to_csv("四格样例.csv", index=False, encoding="utf-8-sig")
print(samp[["格子", "chunk_id", "出处页码", "SFT判定", "RAG判定"]].to_string(index=False))
print("\n=== 第一条完整看一遍 ===")
for k in ("问题", "参考答案", "SFT回答", "RAG回答"):
    print(f"[{k}] {samp.iloc[0][k]}")
cf.download("四格样例.csv")


## 把结果发我

跑完会下载 `对照表.csv`。**五行数字**就是这次实验的全部结论：

| 方案 | 答对率 | 引用正确率 |
| --- | --- | --- |
| 微调后 · 训练事实 · 原问法 | ? | ? |
| 微调后 · 训练事实 · 换问法 | ? | ? |
| 微调后 · 未见事实 · 原问法 | ? | ? |
| 微调后 · 未见事实 · 换问法 | ? | ? |
| A 基线（BM25 + 基座） | ? | ? |

预期：**训练事实上明显涨、换问法还能接住一部分、未见事实仍然接近 0** —— 这就是 SFT 的能与界。
